IMPORTS & CONFIGURATION

In [32]:
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import sqlite3
import plotly.graph_objects as go
from faker import Faker

In [33]:
# Set seed for reproducibility
fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

NUM_USERS = 2500

GENERATE SYNTHETIC DATASETS

In [34]:
countries = ['UK', 'PL', 'PT', 'UAE', 'ES']
channels = ['paid_ad', 'organic', 'referral', 'influencer']

users = []
base_date = datetime(2026, 1, 1)

# Generate Users Table
for i in range(NUM_USERS):
    user_id = f"usr_{10000 + i}"
    signup_time = base_date + timedelta(
        days=random.randint(0, 90),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )
    users.append({
        "user_id": user_id,
        "signup_timestamp": signup_time,
        "country": random.choice(countries),
        "acquisition_channel": random.choice(channels)
    })

df_users = pd.DataFrame(users)

# Generate Funnel Events Table
events = []

for idx, user in df_users.iterrows():
    curr_time = user['signup_timestamp']
    user_id = user['user_id']

    p_step1 = 1.0   # 100% download
    p_step2 = 0.85  # 85% verify phone
    p_step3 = 0.45  # 45% upload ID (Simulated bottleneck)
    p_step4 = 0.90  # 90% activate once ID passes

    # Step 1: App Download
    events.append({
        "event_id": f"evt_{len(events)+1}",
        "user_id": user_id,
        "event_name": '1_app_download',
        "timestamp": curr_time,
        "status": "success"
    })

    # Step 2: Phone Verification
    if random.random() < p_step2:
        curr_time += timedelta(minutes=random.randint(1, 5))
        events.append({
            "event_id": f"evt_{len(events)+1}",
            "user_id": user_id,
            "event_name": '2_phone_verified',
            "timestamp": curr_time,
            "status": "success"
        })

        # Step 3: ID Verification
        if random.random() < p_step3:
            curr_time += timedelta(minutes=random.randint(3, 20))
            id_status = "success" if random.random() < 0.88 else "failed"
            events.append({
                "event_id": f"evt_{len(events)+1}",
                "user_id": user_id,
                "event_name": '3_id_uploaded',
                "timestamp": curr_time,
                "status": id_status
            })

            # Step 4: Account Activation
            if id_status == "success" and random.random() < p_step4:
                curr_time += timedelta(minutes=random.randint(1, 10))
                events.append({
                    "event_id": f"evt_{len(events)+1}",
                    "user_id": user_id,
                    "event_name": '4_account_activated',
                    "timestamp": curr_time,
                    "status": "success"
                })

df_events = pd.DataFrame(events)

print(f"Dataset generated successfully! Total Users: {len(df_users)} | Total Events: {len(df_events)}")

Dataset generated successfully! Total Users: 2500 | Total Events: 6363


PANDAS FUNNEL ANALYSIS

In [35]:
df_success = df_events[df_events['status'] == 'success']

funnel_counts = df_success.groupby('event_name')['user_id'].nunique().reset_index()
funnel_counts.columns = ['funnel_step', 'user_count']

step_order = ['1_app_download', '2_phone_verified', '3_id_uploaded', '4_account_activated']
funnel_counts['sort_order'] = funnel_counts['funnel_step'].map(lambda x: step_order.index(x))
funnel_counts = funnel_counts.sort_values('sort_order').drop(columns=['sort_order'])

total_downloads = funnel_counts.iloc[0]['user_count']
funnel_counts['overall_conversion_pct'] = round((funnel_counts['user_count'] / total_downloads) * 100, 2)

funnel_counts['previous_step_users'] = funnel_counts['user_count'].shift(1)
funnel_counts['step_dropoff_pct'] = round(
    ((funnel_counts['previous_step_users'] - funnel_counts['user_count']) / funnel_counts['previous_step_users']) * 100, 2
)

print("\n=== ONBOARDING FUNNEL PERFORMANCE ===")
print(funnel_counts[['funnel_step', 'user_count', 'overall_conversion_pct', 'step_dropoff_pct']].to_string(index=False))


=== ONBOARDING FUNNEL PERFORMANCE ===
        funnel_step  user_count  overall_conversion_pct  step_dropoff_pct
     1_app_download        2500                  100.00               NaN
   2_phone_verified        2106                   84.24             15.76
      3_id_uploaded         858                   34.32             59.26
4_account_activated         781                   31.24              8.97


SQL QUERY VIA IN-MEMORY SQLITE

In [36]:
conn = sqlite3.connect(':memory:')
df_users.to_sql('users', conn, index=False, if_exists='replace')
df_events.to_sql('funnel_events', conn, index=False, if_exists='replace')

sql_query = """
WITH funnel_counts AS (
    SELECT
        event_name,
        COUNT(DISTINCT user_id) AS user_count,
        CASE event_name
            WHEN '1_app_download' THEN 1
            WHEN '2_phone_verified' THEN 2
            WHEN '3_id_uploaded' THEN 3
            WHEN '4_account_activated' THEN 4
        END AS step_order
    FROM funnel_events
    WHERE status = 'success'
    GROUP BY event_name
)
SELECT
    event_name,
    user_count,
    ROUND(
        (user_count * 100.0) / FIRST_VALUE(user_count) OVER (ORDER BY step_order),
        2
    ) AS overall_conversion_pct,
    ROUND(
        (user_count * 100.0) / LAG(user_count, 1) OVER (ORDER BY step_order),
        2
    ) AS step_retention_pct
FROM funnel_counts
ORDER BY step_order;
"""

funnel_results = pd.read_sql_query(sql_query, conn)

DETAILED ONBOARDING ANALYSIS

In [37]:
funnel_results_indexed = funnel_results.set_index('event_name').reindex(step_order)

failed_events_df = df_events[df_events['status'] == 'failed']
failed_counts = failed_events_df.groupby('event_name')['user_id'].nunique().reset_index()
failed_counts.columns = ['event_name', 'users_explicitly_failed_step']

funnel_analysis = funnel_results_indexed.merge(failed_counts, on='event_name', how='left').fillna(0)

users_started = [NUM_USERS]
for i in range(1, len(step_order)):
    users_started.append(funnel_analysis['user_count'].iloc[i-1])

funnel_analysis['Users Started'] = users_started
funnel_analysis['Total Drop-off at this Step'] = funnel_analysis['Users Started'] - funnel_analysis['user_count']

final_funnel_numbers = funnel_analysis[[
    'Users Started',
    'user_count',
    'users_explicitly_failed_step',
    'Total Drop-off at this Step'
]].rename(columns={
    'user_count': 'Users Completed Successfully',
    'users_explicitly_failed_step': 'Users Failed at this Step'
}).reset_index().rename(columns={'event_name': 'Funnel Step'})

print("\n=== DETAILED ONBOARDING FUNNEL ANALYSIS ===")
print(final_funnel_numbers.to_string(index=False))


=== DETAILED ONBOARDING FUNNEL ANALYSIS ===
 index  Users Started  Users Completed Successfully  Users Failed at this Step  Total Drop-off at this Step
     0           2500                          2500                        0.0                            0
     1           2500                          2106                        0.0                          394
     2           2106                           858                      118.0                         1248
     3            858                           781                        0.0                           77


VISUALIZATION

In [38]:
fig = go.Figure(go.Funnel(
    y = funnel_results['event_name'],
    x = funnel_results['user_count'],
    textinfo = "value+percent initial",
    marker = {"color": ["deepskyblue", "lightseagreen", "lightcoral", "mediumpurple"]},
    connector = {"line": {"color": "white", "dash": "dot", "width": 3}}
))

fig.update_layout(
    title="Revolut Onboarding Funnel: User Count by Stage",
    xaxis_title="Number of Users",
    yaxis_title="Funnel Stage"
)

fig.show()